# E1 — Model 2 as a classifier (encoder + one head per field)

The generative Model 2 spends its candidate list badly. Five beams of the final configuration
carry 2.84 distinct solvents and 1.18 distinct catalysts: beams diverge on how a formula is
spelled long before they diverge on which substance is named. The strict top-5 pays for it --
37.0% solvent and 39.5% catalyst, *below* the 40.5% and 43.6% of naming the five most frequent
sets of the corpus without reading the reaction at all. Rank 1 tells the other story: 26.6%
against 10.7%, so the model does read the reaction. The defect is the list, not the
discrimination (RESULTS.md, "E0").

A softmax cannot repeat itself. This run replaces the decoder with one linear head per field:

- solvent, catalyst -- 1,000 classes each, a class being the whole component *set*, which is
  what `exact_match` compares. Mixtures need no special handling and their order is gone before
  training starts. Ceiling on the test set: 95.1% and 97.1%.
- temperature -- 200 classes, the *values* themselves (0, 80, 100, -78 ...), not ranges. Ceiling
  99.4% exact, 99.5% within +-10 C. A range would leave `within_tol` uncomputable and hand the
  chemist an interval where the protocol needs a figure.

Incomplete annotation moves from the prefix scheme into the loss: solvent and catalyst get an
explicit "not specified" class (for the catalyst that is a real answer, and it costs one slot of
five at most), while rows without a temperature are masked out of that head's loss entirely --
training on them is what taught the generative runs to abstain.

Corpus and test are F1's, unchanged: 486,330 training rows, the 5,687-row clean test.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))

rows = [json.loads(line) for line in open(train_file)]
test_rows = [json.loads(line) for line in open(test_file)]
print(f"train {len(rows)} | val {sum(1 for _ in open(val_file))} | test {len(test_rows)}")

# Same three assertions F1 ran on the same dataset: the roles split has to have travelled with
# the data, the input column has to be the full charge (the same field name means the
# substrates alone in the pre-roles files), and the test must be the clean 5,687.
assert "full_reactants_smiles" in rows[0], "dataset is the pre-roles one"
assert len(rows) == 486330, f"expected the 486,330-row corpus, got {len(rows)}"
assert len(test_rows) == 5687, "wrong test file"

test_products = {r["product_smiles"] for r in test_rows}
leaks = sum(1 for r in rows if r["product_smiles"] in test_products)
print("rows whose product is in the test:", leaks)
assert leaks == 0, f"{leaks} leaked rows"

base_model = "t5-small"
learning_rate = 5e-4
condition_fields = "solvent,catalyst,temperature_celsius"
num_classes = "solvent=1000,catalyst=1000,temperature_celsius=200"
output_dir = "/kaggle/working/e1_classifier"
time_budget_minutes = 480

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Effective batch 64 on two T4s, F1's batch. No --group-by-length: with no decoder the target
# is one integer per field, so a batch's cost is set by the source alone and bucketing would
# only correlate the class distribution with the batch order.
!torchrun --nproc_per_node=2 scripts/train_conditions_classifier.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_e1_classifier \
    --reactants-field full_reactants_smiles \
    --condition-fields "{condition_fields}" \
    --num-classes "{num_classes}" \
    --max-source-length 256 \
    --per-device-train-batch-size 32 \
    --per-device-eval-batch-size 64 \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --eval-steps 1000 --save-steps 1000 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done or paused at the budget; tail of log:")
!tail -8 "{log_path}"

In [ ]:
import json

meta = json.load(open(f"{output_dir}/final/classifier_meta.json"))
print("reactant side:", meta["reactants_field"], "| fields:", meta["fields"])
assert meta["reactants_field"] == "full_reactants_smiles", "trained on the substrates-only column"

for field, vocabulary in meta["vocabularies"].items():
    print(
        f"  {field}: {len(vocabulary['classes'])} classes, covering "
        f"{vocabulary['coverage_of_named_rows']:.1%} of the {vocabulary['train_rows_with_value']} "
        f"rows that name one ({vocabulary['train_rows_absent']} absent)"
    )

# Absence is an answer for the substance fields and a hole in the protocol for temperature.
assert meta["vocabularies"]["catalyst"]["absent_class"] == 0, "catalyst lost its silence class"
assert meta["vocabularies"]["temperature_celsius"]["absent_class"] is None, \
    "temperature must be masked, not taught to abstain"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print(f"  last: {points[-1]} | best: {state.get('best_metric')}")
print(f"  reached epoch {points[-1][0]:.2f} of 3")

In [ ]:
!python scripts/evaluate_conditions_classifier.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --top-k 5 --batch-size 64 --device cuda \
    --output "/kaggle/working/E1_classifier_clean_topk.json"

In [ ]:
import json

# The three runs E1 has to be read against, all on the same 5,687-row clean test and the same
# strict top-5: the generative final configuration, that configuration with its candidate list
# backfilled from the corpus prior (E0), and the prior alone.
REFERENCE = {
    "solvent_exact_match_top5": ("F1 37.0", "F1+prior 45.7", "prior 40.5"),
    "solvent_same_group_top5": ("F1 54.8", "F1+prior 67.0", "prior 54.8"),
    "catalyst_exact_match_top5": ("F1 39.5", "F1+prior 55.3", "prior 43.6"),
    "catalyst_same_group_top5": ("F1 58.0", "F1+prior 73.1", "prior 60.9"),
    "catalyst_record_level_top5": ("F1 84.1", "F1+prior 86.7", "prior 7.2"),
    "temperature_celsius_within_tol_top5": ("F1 67.0", "F1+prior 67.0", "prior --"),
    "temperature_celsius_same_bucket_top5": ("F1 76.2", "F1+prior 76.2", "prior --"),
}

summary = json.load(open("/kaggle/working/E1_classifier_clean_topk.json"))["summary"]
print(f'{"metric":<40}{"E1":>8}{"F1":>12}{"F1+prior":>16}{"prior":>12}')
for key, (f1, fill, prior) in REFERENCE.items():
    print(f"{key:<40}{summary.get(key, 0) * 100:>7.1f}%{f1:>12}{fill:>16}{prior:>12}")

# Denominators must match the generative runs exactly, or the columns above compare different
# populations.
print("\ndenominators:", {k: v for k, v in summary.items() if k.endswith("_expected_count")})
assert summary["solvent_expected_count"] == 5077
assert summary["catalyst_expected_count"] == 941
assert summary["temperature_celsius_expected_count"] == 1579

# The point of the reformulation: five slots, five different answers.
records = json.load(open("/kaggle/working/E1_classifier_clean_topk.json"))["records"]
distinct = [len({c.split("|")[0] for c in r["candidates_raw"]}) for r in records]
print(f"distinct solvents among the 5 candidates: {sum(distinct) / len(distinct):.2f} (F1: 2.84)")